# Analyse an Opta file with `wa-setpieces`

This notebook loads one Opta JSON export, validates it, runs the complete set-piece workflow, displays the results, and exports CSV tables plus an HTML report.

In [ ]:
# Run once when using the repository checkout:
%pip install -e "..[viz,ml,convert]"

In [ ]:
from pathlib import Path
import pandas as pd

from wa_setpieces import (
    XTModel,
    defensive_rating,
    load_events,
    run_workflow,
    validate_events,
    write_html_report,
)

## Configuration

Change `OPTA_FILE` to your Opta JSON export. For production added-value ratings, point `XT_MODEL_FILE` to an xT model trained on a season or larger sample.

In [ ]:
OPTA_FILE = Path("../tests/data/sample_match.json")
SET_PIECE_TYPE = "corner"  # corner, free_kick, throw_in, goal_kick, kick_off, penalty
OUTPUT_DIR = Path("../analysis")
XT_MODEL_FILE = None  # Example: Path("../league_xt.npz")
FIT_XT_ON_THIS_MATCH = False  # Illustrative only; a single match is too small for production xT

## Load and validate the Opta events

In [ ]:
match = load_events(OPTA_FILE)
events = validate_events(match.events)

print(f"Loaded {len(events):,} events")
print(f"Teams: {events['contestantId'].nunique()}")
events.head()

## Load or fit the xT model

In [ ]:
model = None
if XT_MODEL_FILE is not None:
    model = XTModel.load(XT_MODEL_FILE)
    print("Loaded xT model:", model.metadata)
elif FIT_XT_ON_THIS_MATCH:
    model = XTModel.fit(events)
    print("Warning: xT was fitted on one match and is illustrative only.")
else:
    print("Running without xT. Added-value and player-rating tables will be unavailable.")

## Run the complete workflow

In [ ]:
result = run_workflow(events, SET_PIECE_TYPE, model=model)
result.summary

## Inspect the available tables

In [ ]:
tables = {
    name: value
    for name, value in vars(result).items()
    if isinstance(value, pd.DataFrame)
}

if not result.defensive_summary.empty:
    tables["defensive_rating"] = defensive_rating(result.defensive_summary)

pd.DataFrame(
    [{"table": name, "rows": len(table), "columns": len(table.columns)}
     for name, table in tables.items()]
)

In [ ]:
# Examples of individual outputs:
display(result.report)
display(result.defensive_summary)
display(result.routine_summary)
display(result.routine_team_profiles)
display(result.routine_taker_profiles)
display(result.routine_target_matrix)
display(result.routines.head())
display(result.first_contacts.head())

## Export CSV tables and an HTML report

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for name, table in tables.items():
    table.to_csv(OUTPUT_DIR / f"{name}.csv", index=False)

report_path = write_html_report(
    OUTPUT_DIR / "report.html",
    title=f"{SET_PIECE_TYPE.replace('_', ' ').title()} analysis",
    tables=tables,
    methodology=(
        "Source: Opta event data. Retention, phases, first contact and added value "
        "are derived event-data heuristics. Ratings need a season-sized benchmark."
    ),
)

print(f"Wrote {len(tables)} CSV files")
print(f"HTML report: {report_path.resolve()}")